In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Chybová mitigace pomocí tenzorových sítí (TEM): Qiskit funkce od Algorithmiq
> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (přes IBM Quantum Platform API) Plan. Jsou ve stavu preview release a mohou se změnit.


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Přehled
Metoda chybové mitigace pomocí tenzorových sítí (TEM) od Algorithmiq je hybridní
kvantově-klasický algoritmus navržený pro provádění mitigace šumu výhradně ve
fázi klasického post-processingu. S TEM lze vypočítat střední hodnoty
pozorovatelných veličin a zmírnit nevyhnutelné chyby způsobené šumem na
kvantovém hardware s vyšší přesností a efektivitou nákladů,
což z něj dělá velmi atraktivní volbu pro kvantové výzkumníky i průmyslové
odborníky.

Metoda spočívá v sestavení tenzorové sítě reprezentující inverzi
globálního kanálu šumu ovlivňujícího stav kvantového procesoru a následném
aplikování tohoto zobrazení na informačně úplné výsledky měření získané
z zašuměného stavu, čímž se získají nestranné odhadce pozorovatelných veličin.

Jako výhodu TEM využívá informačně úplná měření pro přístup k rozsáhlé sadě
mitigovaných středních hodnot pozorovatelných veličin a dosahuje optimální
vzorkovací režie na kvantovém hardware, jak je popsáno ve Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), a Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Vzorkovací
režie označuje počet dodatečných měření potřebných k provedení efektivní
chybové mitigace, což je klíčový faktor pro proveditelnost kvantových výpočtů.
TEM má proto potenciál umožnit kvantovou výhodu ve složitých scénářích, jako jsou
aplikace v oblasti kvantového chaosu, fyziky mnoha těles, Hubbardovy dynamiky
a simulací chemie malých molekul.

Hlavní vlastnosti a výhody TEM lze shrnout takto:

1.  **Optimální vzorkovací režie**: TEM je optimální vzhledem k
[teoretickým hranicím](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
což znamená, že žádná metoda nemůže dosáhnout menší vzorkovací režie. Jinými
slovy, TEM vyžaduje minimální počet dodatečných měření pro provedení
chybové mitigace. To zároveň znamená, že TEM používá minimální kvantový runtime.
2.  **Nákladová efektivita**: Protože TEM zpracovává mitigaci šumu výhradně ve
fázi post-processingu, není potřeba přidávat do kvantového počítače extra Circuit,
což nejen snižuje náklady na výpočet, ale také snižuje riziko zavlečení
dodatečných chyb způsobených nedokonalostmi kvantových zařízení.
3.  **Odhad více pozorovatelných veličin**: Díky informačně úplným měřením TEM
efektivně odhaduje více pozorovatelných veličin ze stejných dat měření získaných
z kvantového počítače.
4.  **Mitigace chyb měření**: Qiskit Function TEM také zahrnuje
[proprietární metodu mitigace chyb měření](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
která dokáže výrazně snížit chyby čtení po krátkém kalibračním běhu.
5.  **Přesnost**: TEM výrazně zlepšuje přesnost a spolehlivost digitálních
kvantových simulací, čímž dělá kvantové algoritmy přesnějšími a spolehlivějšími.
## Popis
Funkce TEM ti umožňuje získat mitigované střední hodnoty pro
více pozorovatelných veličin na kvantovém Circuit s minimální vzorkovací režií.
Circuit je měřen pomocí informačně úplného měření s pozitivně semidefinitní
operátorovou hodnotou (IC-POVM) a získané výsledky měření jsou zpracovány
na klasickém počítači. Toto měření se používá k provedení metod tenzorových sítí
a sestavení mapy inverze šumu. Funkce aplikuje zobrazení, které plně invertuje
celý zašuměný Circuit pomocí tenzorových sítí reprezentujících zašuměné vrstvy.

![Schéma TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Mitigovaný odhad pozorovatelné O pomocí post-processingu výsledků měření zašuměného kvantového procesoru. U a N označují ideální kvantovou operaci a přidružené zobrazení šumu, které může být obecně nelokální (rozšířeno na šedé boxy). D označuje tenzor operátorů duálních k efektům v IC měření. Modul mitigace šumu M je tenzorová síť efektivně kontrahovaná od středu. První iterace kontrakce je znázorněna tečkovanou fialovou čarou, druhá přerušovanou čarou a třetí plnou čarou.")

Jakmile jsou Circuit odeslány do funkce, jsou transpilovaný a
optimalizovány tak, aby se minimalizoval počet vrstev s dvou-Qubitovými Gate
(nejhlučnějšími Gate na kvantových zařízeních). Šum ovlivňující vrstvy je
naučen prostřednictvím
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
pomocí řídkého Pauli-Lindbladova modelu šumu, jak je popsáno v E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model šumu je přesný popis šumu na zařízení schopný zachytit jemné rysy,
včetně přeslechů Qubitů. Šum na zařízeních však může kolísat a driftovat
a naučený šum nemusí být přesný v okamžiku provádění odhadu. To může vést
k nepřesným výsledkům.
## Začínáme
Proveď autentizaci pomocí svého [klíče IBM Quantum Platform API](http://quantum.cloud.ibm.com/) a vyber funkci TEM takto. (Tento úryvek předpokládá, že sis již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do lokálního prostředí.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Příklad
Následující úryvek ukazuje příklad, kde TEM slouží k výpočtu středních hodnot pozorovatelné veličiny pro jednoduchý kvantový Circuit.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Ke kontrole stavu tvé Qiskit Function pracovní zátěže použij Qiskit Serverless API:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Výsledky lze vrátit takto:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Očekávaná hodnota pro bezšumový Circuit pro daný operátor by měla být přibližně `0.18409094298943401`.
## Vstupy
**Parametry**

Název | Typ | Popis | Povinný | Výchozí | Příklad
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Iterovatelná kolekce PUB-like (primitive unified bloc) objektů, jako jsou n-tice `(circuit, observables)` nebo `(circuit, observables, parameters, precision)`. Více informací najdeš v části [Přehled PUBů](/guides/primitive-input-output#overview-of-pubs). Pokud je předán ne-ISA Circuit, bude transpilován s optimálním nastavením. Pokud je předán ISA Circuit, nebude transpilován; v tomto případě musí být pozorovatelná definována na celém QPU. | Ano | N/A | (circuit, observables)
`backend_name` | str | Název Backend pro zpracování dotazu.| Ne | Pokud není zadán, bude použit nejméně vytížený Backend. | "ibm_fez"
`options` | dict | Vstupní možnosti. Více podrobností najdeš v části `Options`. | Ne | Viz část `Options` pro výchozí hodnoty.| {"max_bond_dimension": 100\}

> **Caution:** TEM má v současnosti následující omezení:
> 
>   - Parametrizované Circuit nejsou podporovány. Argument parameters by měl být nastaven na `None`, pokud je zadána přesnost. Toto omezení bude v budoucích verzích odstraněno.
>   - Podporovány jsou pouze Circuit bez smyček. Toto omezení bude v budoucích verzích odstraněno.
>   - Neunitární Gate, jako je reset, measure a všechny formy řízení toku, nejsou podporovány. Podpora pro reset bude přidána v nadcházejících verzích.
### Možnosti
Slovník obsahující pokročilé možnosti pro TEM. Slovník může obsahovat klíče z následující tabulky. Pokud některá z možností není zadána, bude použita výchozí hodnota uvedená v tabulce. Výchozí hodnoty jsou vhodné pro typické použití TEM.

Název | Volby | Popis  | Výchozí
-- | -- | -- | --
`tem_max_bond_dimension` | int | Maximální dimenze vazby používaná pro tenzorové sítě. | 500 |
`tem_compression_cutoff` | float | Hodnota cutoff používaná pro tenzorové sítě. | 1e-16
`compute_shadows_bias_from_observable` | bool | Booleovský příznak indikující, zda má být bias pro protokol měření klasických stínů přizpůsoben PUB pozorovatelné nebo ne. Pokud je False, bude použit protokol klasických stínů (stejná pravděpodobnost měření Z, X, Y).| False
`shadows_bias` | np.ndarray | Bias používaný pro randomizovaný protokol měření klasických stínů, 1D nebo 2D pole velikosti 3 nebo tvaru (num_qubits, 3). Pořadí je ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int nebo `None` | Maximální doba provádění na QPU v sekundách. Pokud runtime překročí tuto hodnotu, úloha bude zrušena. Pokud je `None`, použije se výchozí limit nastavený Qiskit Runtime. | `None`
`num_randomizations` | int | Počet randomizací používaných pro učení šumu a twirling Gate. | 32
`max_layers_to_learn` | int | Maximální počet unikátních vrstev k naučení. | 4
`mitigate_readout_error` | bool | Booleovský příznak indikující, zda provádět mitigaci chyb čtení nebo ne. | True
`num_readout_calibration_shots` | int | Počet vzorků používaných pro mitigaci chyb čtení. | 10000
`default_precision` | float | Výchozí přesnost používaná pro PUB, pro které přesnost není zadána. |0.02
`seed` | int nebo `None` | Nastaví seed generátoru náhodných čísel pro reprodukovatelnost. Pokud je `None`, seed se nenastaví. | None
## Výstupy
Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) obsahující TEM-mitigovaný výsledek. Výsledek pro každý PUB je vrácen jako [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) obsahující následující pole:

Název |Typ | Popis
-- | -- | --
data | DataBin | Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) obsahující TEM-mitigovanou pozorovatelnou a její standardní chybu. DataBin má následující pole: <ul><li>`evs`: Hodnota TEM-mitigované pozorovatelné.</li><li>`stds`: Standardní chyba TEM-mitigované pozorovatelné.</li></ul>
metadata | dict | Slovník obsahující dodatečné výsledky. Slovník obsahuje následující klíče: <ul><li>`"evs_non_mitigated"`: Hodnota pozorovatelné bez chybové mitigace.</li><li>`"stds_non_mitigated"`: Standardní chyba výsledku bez chybové mitigace.</li><li>`"evs_mitigated_no_readout_mitigation"`: Hodnota pozorovatelné s chybovou mitigací, ale bez mitigace chyb čtení.</li><li>`"stds_mitigated_no_readout_mitigation"`: Standardní chyba výsledku s chybovou mitigací, ale bez mitigace chyb čtení.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Hodnota pozorovatelné bez chybové mitigace, ale s mitigací chyb čtení.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Standardní chyba výsledku bez chybové mitigace, ale s mitigací chyb čtení.</li></ul>
## Načítání chybových zpráv
Pokud je stav tvé pracovní zátěže ERROR, načti chybovou zprávu pomocí job.result() takto: